In [3]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import AgentMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# 加载环境变量
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# 初始化模型
model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [4]:
@tool
def get_weather(city: str) -> str:
    """查询城市天气"""
    weather_data = {
        "北京": "晴天，15°C",
        "上海": "多云，18°C",
        "深圳": "雨天，22°C"
    }
    return weather_data.get(city, "未知城市")

In [38]:
class LoggingMiddleware(AgentMiddleware): # ✅ 类名随意
    """
    日志中间件 - 记录每次模型调用

    before_model: 模型调用前执行
    after_model: 模型响应后执行
    """

    def before_model(self, state, runtime):
        """模型调用前"""
        print("\n[中间件] before_model: 准备调用模型")
        print(f"[中间件] 当前消息数: {len(state.get('messages', []))}")

        print("-"*50)
        print("state"   ,state)
        print("runtime  ",runtime)
        print("state['messages']    ",state["messages"])
        print("state.get('messages')    ",state.get("messages"))
        print("state.get('messages')[-1]    ",state.get("messages")[-1])
        print("state.get('messages')[-1].content    ",state.get("messages")[-1].content)
        print("-"*50)
        return None  # 返回 None 表示继续正常流程

    def after_model(self, state, runtime):
        """模型响应后"""
        print("[中间件] after_model: 模型已响应")
        last_message = state.get('messages', [])[-1]
        print(f"[中间件] 响应类型: {last_message.__class__.__name__}")
        
        return None  # 返回 None 表示不修改状态

In [39]:
"""
示例1：基础中间件 - 日志记录

展示 before_model 和 after_model 的基本用法
"""
print("\n" + "="*70)
print("示例 1：基础中间件 - 日志记录")
print("="*70)

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
    middleware=[LoggingMiddleware()]  # 添加中间件
)

print("\n用户: 你好")
response = agent.invoke({"messages": [{"role": "user", "content": "你好"}]})
print(f"Agent: {response['messages']}")
print(f"Agent: {response['messages'][-1].content}")



示例 1：基础中间件 - 日志记录

用户: 你好

[中间件] before_model: 准备调用模型
[中间件] 当前消息数: 1
--------------------------------------------------
state {'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='0fc9f4fa-a9fb-4dd3-b5a4-61b290434230')]}
runtime   Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x10d921e40>, previous=None, execution_info=ExecutionInfo(node_attempt=1, node_first_attempt_time=1775117009.205337))
state['messages']     [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='0fc9f4fa-a9fb-4dd3-b5a4-61b290434230')]
state.get('messages')     [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='0fc9f4fa-a9fb-4dd3-b5a4-61b290434230')]
state.get('messages')[-1]     content='你好' additional_kwargs={} response_metadata={} id='0fc9f4fa-a9fb-4dd3-b5a4-61b290434230'
state.get('messages')[-1].content     你好
--------------------------------------------------
[中间件] after_model:

In [40]:
# 修改状态的中间件
class CallCounterMiddleware(AgentMiddleware):
    """
    计数中间件 - 统计模型调用次数

    在中间件内部维护计数器（简单版本）
    """

    def __init__(self):
        super().__init__()
        self.count = 0  # 简单计数器

    def after_model(self, state, runtime):
        """模型响应后，增加计数"""
        self.count += 1
        print(f"\n[计数器] 模型调用次数: {self.count}")
        return None  # 不修改 state

In [44]:
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
    middleware=[CallCounterMiddleware()],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "counter_test"}}

print("\n第一次调用:")
response=agent.invoke({"messages": [{"role": "user", "content": "你好"}]}, config)
print(response["messages"][-1].content)

print("\n第二次调用:")
response1=agent.invoke({"messages": [{"role": "user", "content": "今天天气"}]}, config)
print(response1["messages"][-1].content)

print("\n第三次调用:")
response2 = agent.invoke({"messages": [{"role": "user", "content": "谢谢"}]}, config)
print(response2["messages"][-1].content)

print(response2["messages"])
print(len(response2["messages"]))
for i in range(len(response2["messages"])):
    print(f"第{i}次：   {response2['messages'][i]}")


第一次调用:

[计数器] 模型调用次数: 1
你好。你需要帮助或只是想聊天吗？

第二次调用:

[计数器] 模型调用次数: 2
我很乐意与您聊聊天气！不过，我是一个大型语言模型，我没有实时访问当前天气条件的权限。但是我可以建议一些方法来找到您所在地区的天气信息。

您可以查看天气应用程序或网站，例如AccuWeather、Weather.com或国家气象局，获取当地天气预报、温度和状况的最新信息。您也可以在手机或电脑上输入“当前位置天气”来获取即时更新。

如果您有特定的位置或城市，我可以尝试为您提供一般的天气信息或历史天气模式。这对您有帮助吗？

第三次调用:

[计数器] 模型调用次数: 3
不客气！很高兴能帮到您。如果您需要任何其他帮助或只是想聊天，请随时问我。祝您有美好的一天！
[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='ea30f34e-5c68-4bb0-9c09-29c4f9f0f96c'), AIMessage(content='你好。你需要帮助或只是想聊天吗？', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 45, 'total_tokens': 58, 'completion_time': 0.071594576, 'completion_tokens_details': None, 'prompt_time': 0.006195089, 'prompt_tokens_details': None, 'queue_time': 0.111056154, 'total_time': 0.077789665}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--

In [17]:
class MessageTrimmerMiddleware(AgentMiddleware):
    """
    消息修剪中间件 - 限制消息数量

    before_model 修改消息列表
    注意：需要配合无 checkpointer 使用，否则历史会被恢复
    """

    def __init__(self, max_messages=5):
        super().__init__()
        self.max_messages = max_messages
        self.trimmed_count = 0  # 统计修剪次数

    def before_model(self, state, runtime):
        """模型调用前，修剪消息"""
        messages = state.get('messages', [])

        if len(messages) > self.max_messages:
            # 保留最近的 N 条消息
            trimmed_messages = messages[-self.max_messages:]
            self.trimmed_count += 1
            print(f"\n[修剪] 消息从 {len(messages)} 条减少到 {len(trimmed_messages)} 条 (第{self.trimmed_count}次修剪)")
            return {"messages": trimmed_messages}

        return None


In [18]:
middleware = MessageTrimmerMiddleware(max_messages=4)  # 最多保留 4 条
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
    middleware=[middleware]
    # 不使用 checkpointer
)

# 手动管理消息历史
messages = []
for i in range(6):
    print(f"\n--- 第 {i+1} 次对话 ---")

    # 新增用户消息
    new_msg = {"role": "user", "content": f"消息{i+1}：简短回复"}
    messages.append(new_msg)

    print(f"调用前消息数: {len(messages)}")

    # 调用 agent（middleware会修剪）
    response = agent.invoke({"messages": messages})

    # 获取完整对话（包含AI响应）
    messages = response['messages']

    print(f"调用后消息数: {len(messages)}")
    if len(messages) <= 4:
        print(f"消息列表: {[m.content[:15] for m in messages]}")

print(f"\n修剪统计: 共修剪了 {middleware.trimmed_count} 次")




--- 第 1 次对话 ---
调用前消息数: 1
调用后消息数: 2
消息列表: ['消息1：简短回复', '我能帮你什么？']

--- 第 2 次对话 ---
调用前消息数: 3
调用后消息数: 4
消息列表: ['消息1：简短回复', '我能帮你什么？', '消息2：简短回复', '请随时提出问题。']

--- 第 3 次对话 ---
调用前消息数: 5

[修剪] 消息从 5 条减少到 4 条 (第1次修剪)
调用后消息数: 6

--- 第 4 次对话 ---
调用前消息数: 7

[修剪] 消息从 7 条减少到 4 条 (第2次修剪)
调用后消息数: 8

--- 第 5 次对话 ---
调用前消息数: 9

[修剪] 消息从 9 条减少到 4 条 (第3次修剪)
调用后消息数: 10

--- 第 6 次对话 ---
调用前消息数: 11

[修剪] 消息从 11 条减少到 4 条 (第4次修剪)
调用后消息数: 12

修剪统计: 共修剪了 4 次


In [45]:
class Middleware1(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件1] before_model")
        return None
    def after_model(self, state, runtime):
        print("[中间件1] after_model")
        return None

class Middleware2(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件2] before_model")
        return None
    def after_model(self, state, runtime):
        print("[中间件2] after_model")
        return None

class Middleware3(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件3] before_model")
        return None
    def after_model(self, state, runtime):
        print("[中间件3] after_model")
        return None

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
    middleware=[Middleware1(), Middleware2(), Middleware3()]
)

print("\n执行一次调用，观察顺序：")
agent.invoke({"messages": [{"role": "user", "content": "测试"}]})




执行一次调用，观察顺序：
[中间件1] before_model
[中间件2] before_model
[中间件3] before_model
[中间件3] after_model
[中间件2] after_model
[中间件1] after_model


{'messages': [HumanMessage(content='测试', additional_kwargs={}, response_metadata={}, id='cd1d8dff-c340-48e2-ba2f-d96d8d073b7f'),
  AIMessage(content='测试通过。您好，我可以帮您做些什么吗？', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 44, 'total_tokens': 59, 'completion_time': 0.112471973, 'completion_tokens_details': None, 'prompt_time': 0.006709466, 'prompt_tokens_details': None, 'queue_time': 0.185744761, 'total_time': 0.119181439}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_e65acd3773', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4d3e-55be-7d61-a4cf-923aa7623a16-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 15, 'total_tokens': 59})]}

In [49]:
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            max_tokens_before_summary=200  # 超过 200 token 就摘要
        )
    ],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "summary_test"}}

# 连续多轮对话
conversations = [
    "介绍一下 Python",
    "它有哪些特点？",
    "主要应用在哪些领域？",
    "和 Java 的区别是什么？"
]

for i, msg in enumerate(conversations):
    print(f"\n--- 第 {i+1} 轮对话 ---")
    print(f"用户: {msg}")
    response = agent.invoke({"messages": [{"role": "user", "content": msg}]}, config)
    print(f"Agent: {response['messages'][-1].content[:100]}...")
    print(f"总消息数: {len(response['messages'])}")


/var/folders/zt/63tm27h17v75fy_47rqppsl40000gn/T/ipykernel_81260/3076092230.py:8: DeprecationWarning: max_tokens_before_summary is deprecated. Use trigger=('tokens', value) instead.
  SummarizationMiddleware(



--- 第 1 轮对话 ---
用户: 介绍一下 Python
Agent: Python是一种高级、解释型编程语言，于1991年由Guido van Rossum首次发布。它的设计目标是易于学习、简单直观，强调代码的可读性和简洁性。Python支持面向对象、命令式、函数式和过...
总消息数: 2

--- 第 2 轮对话 ---
用户: 它有哪些特点？
Agent: Python具有以下特点：

1. **易于学习**：Python的语法简单清晰，适合初学者学习。
2. **高级语言**：Python是一种高级语言，意味着它抽象了许多底层细节，允许开发者专注于编程...
总消息数: 4

--- 第 3 轮对话 ---
用户: 主要应用在哪些领域？
Agent: Python的主要应用领域包括：

1. **数据分析和科学计算**：Python广泛用于数据分析、科学计算和数据可视化，尤其是在物理、工程、经济和金融等领域。流行的库包括NumPy、Pandas、M...
总消息数: 6

--- 第 4 轮对话 ---
用户: 和 Java 的区别是什么？
Agent: Python 和 Java 是两种流行的编程语言，但它们在语法、特性和用例方面有着显著的差异。以下是 Python 和 Java 的一些主要区别：

1. **语法**：Python 的语法比 Jav...
总消息数: 8
